### Import Required Libraries

In [2]:
import pandas as pd
import os
import glob
from datetime import datetime
import re

In [6]:

# === CONFIGURATION ===
input_folder = r"D:\Nilesh\Telangana Processing\2024 Telangana\Rate_Finalize"

# === FIND ALL EXCEL FILES IN FOLDER ===
excel_files = [f for f in os.listdir(input_folder) if f.endswith('.xlsx')]

# === READ AND CONCATENATE ===
merged_df = pd.DataFrame()
total_records = 0
file_counts = {}

print("=" * 60)
print("MERGING EXCEL FILES")
print("=" * 60)

for file in sorted(excel_files):
    file_path = os.path.join(input_folder, file)
    
    # Read the file
    temp_df = pd.read_excel(file_path)
    
    # Store count for this file
    file_counts[file] = len(temp_df)
    total_records += len(temp_df)
    
    # Display count for current file
    print(f"📄 {file}: {len(temp_df):,} records")
    
    # Merge
    merged_df = pd.concat([merged_df, temp_df], ignore_index=True)

# === DISPLAY SUMMARY ===
print("=" * 60)
print(f"📊 TOTAL FILES PROCESSED: {len(excel_files)}")
print(f"📊 TOTAL RECORDS IN MERGED DATAFRAME: {len(merged_df):,}")

# Optional: Display individual file counts in a table
print("\n📋 DETAILED FILE COUNTS:")
print("-" * 60)
for file, count in file_counts.items():
    print(f"{file:<50} {count:>10,} records")
print("-" * 60)
print(f"{'TOTAL':<50} {total_records:>10,} records")
print("=" * 60)

# === VERIFICATION ===
if total_records == len(merged_df):
    print("✅ Record count verification: MATCHED")
else:
    print("⚠️ Warning: Record count mismatch!")

# merged_df is now available for further cleaning
print("\n✅ Merging complete! 'merged_df' is ready for next processing steps.")
print(f"DataFrame shape: {merged_df.shape}")

MERGING EXCEL FILES
📄 Hydrabad_Final(Manual_Done_By_Nilesh).xlsx: 62,560 records
📄 Medchal_Makajgiri_Final(Manual_Done_Nilesh).xlsx: 151,447 records
📄 RangaReddy_Final.xlsx: 247,471 records
📄 SangaReddy_Final(Manual_Done_By_Aditi).xlsx: 75,537 records
📄 Yadadri_Bhuvangiri_Final(Manual_Done_By_Nilesh).xlsx: 45,928 records
📊 TOTAL FILES PROCESSED: 5
📊 TOTAL RECORDS IN MERGED DATAFRAME: 582,943

📋 DETAILED FILE COUNTS:
------------------------------------------------------------
Hydrabad_Final(Manual_Done_By_Nilesh).xlsx             62,560 records
Medchal_Makajgiri_Final(Manual_Done_Nilesh).xlsx      151,447 records
RangaReddy_Final.xlsx                                 247,471 records
SangaReddy_Final(Manual_Done_By_Aditi).xlsx            75,537 records
Yadadri_Bhuvangiri_Final(Manual_Done_By_Nilesh).xlsx     45,928 records
------------------------------------------------------------
TOTAL                                                 582,943 records
✅ Record count verification: MATCHED

In [7]:
merged_df.to_excel("Telangana_Final.xlsx")

In [2]:
merged_df=pd.read_excel(r"D:\Nilesh\Telangana Processing\2024\Medchal Processing\Medchal_Malkajgiri_With_Property_Type.xlsx")
merged_df.head(2)

## Drop Duplicates

In [3]:
# === SIMPLE VERSION - DROP DUPLICATES ACROSS ALL COLUMNS ===

# Define columns
cols = [
    'S.No.', 'Description of property', 'Reg.Date Exe.Date Pres.Date',
    'Nature & Mkt.Value Con. Value', 'Name of Parties Executant(EX) & Claimants(CL)',
    'Vol/Pg No CD No Doct No/Year', 'Document No', 'District', 'Sub-Registrar Office'
]

# Original count
orig = len(merged_df)
print(f"Original: {orig:,} records")

# Drop duplicates
df_cleaned = merged_df.drop_duplicates(subset=cols, keep='first')

# Result
cleaned = len(df_cleaned)
print(f"Cleaned: {cleaned:,} records")
print(f"Removed: {orig - cleaned:,} duplicate records")
print(f"Unique records: {cleaned/orig*100:.2f}% of original")

Original: 63,190 records
Cleaned: 63,190 records
Removed: 0 duplicate records
Unique records: 100.00% of original


In [3]:
df_cleaned.info()

NameError: name 'df_cleaned' is not defined

## Drop Rows With Rows with only '-' and Rows with 'W-B: 0-0'

In [5]:
df_clean = df_cleaned.copy()

# Convert the column to string type and strip whitespace
descriptions = df_clean['Description of property'].astype(str).str.strip()

# Pattern to match rows that contain only a single hyphen "-" OR exactly "W-B: 0-0"
# Option 1: ^\s*-\s*$ - matches only a single hyphen with possible whitespace
# Option 2: ^\s*W-B:\s*0-0\s*$ - matches exactly "W-B: 0-0" with possible whitespace
pattern = r'^\s*-\s*$|^\s*W-B:\s*0-0\s*$'

# Find rows that match either pattern
rows_to_delete = descriptions.str.contains(pattern, na=False, regex=True)

# Count how many rows will be deleted
deleted_count = rows_to_delete.sum()
total_rows_before = len(df_clean)

# Display the rows that will be deleted
print("Rows that will be deleted:")
print("=" * 50)
deleted_rows = df_clean[rows_to_delete]
if len(deleted_rows) > 0:
    # for idx, row in deleted_rows.iterrows():
    #     print(f"Index {idx}: '{row['Description of property']}'")
    
    # Count by pattern type
    hyphen_mask = descriptions.str.contains(r'^\s*-\s*$', na=False, regex=True)
    wb_mask = descriptions.str.contains(r'^\s*W-B:\s*0-0\s*$', na=False, regex=True)
    
    print(f"\nBreakdown:")
    print(f"Rows with only '-': {hyphen_mask.sum()}")
    print(f"Rows with 'W-B: 0-0': {wb_mask.sum()}")
else:
    print("No rows found matching the deletion criteria")

print(f"\nTotal rows to delete: {deleted_count}")
print(f"Total rows before deletion: {total_rows_before}")

# Ask for confirmation (optional)
confirm = input("\nDo you want to proceed with deletion? (yes/no): ")
if confirm.lower() == 'yes':
    # Keep only rows that don't match either pattern
    df_clean = df_clean[~rows_to_delete].copy()
    df_clean.reset_index(drop=True, inplace=True)
    print(f"\nDeletion completed. Rows remaining: {len(df_clean)}")
else:
    print("Deletion cancelled.")

Rows that will be deleted:

Breakdown:
Rows with only '-': 0
Rows with 'W-B: 0-0': 630

Total rows to delete: 630
Total rows before deletion: 63190

Deletion completed. Rows remaining: 62560


In [6]:
df_clean.shape

(62560, 9)

In [ ]:

# Party type mappings (Seller types, Buyer types)
SELLER_TYPES = {"EX", "MR", "DR", "RR", "PL", "LR","FP"}  # Added PL and LR as Sellers
BUYER_TYPES = {"CL", "ME", "DE", "RE", "AY", "LE","SP"}   # Added AY and LE as Buyers
ALL_PARTY_TYPES = SELLER_TYPES | BUYER_TYPES  # Union of all types

BOUND_START_RE = re.compile(r"(?i)\bbound\w*\s*:\s*")

def clean_spaces(s: str) -> str:
    """Clean extra spaces but preserve intentional newlines"""
    if not isinstance(s, str):
        return s
    # Split into lines, clean each line, then rejoin
    lines = s.split('\n')
    cleaned_lines = [re.sub(r"\s+", " ", line).strip() for line in lines]
    return "\n".join(cleaned_lines)

def normalize_basic(s: str) -> str:
    s = str(s)
    s = re.sub(r"(?i)VILL\s*/\s*COL", "VILL/COL", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def pick_unit_token(v: str, field_name: str = "") -> str:
    if not v:
        return ""
    
    # For BUILT field, return the full value as is (including unit)
    if field_name == "BUILT":
        return v.strip()
    
    # For EXTENT field, keep the original logic
    for p in v.split()[:12]:
        if "SQ" in p.upper():
            return p
    return v

def convert_extent_to_sq_ft(extent_value: str) -> str:
    """Convert extent value from SQ.Yd to SQ.Ft (direct conversion)"""
    if pd.isna(extent_value) or not isinstance(extent_value, str) or not extent_value:
        return ""
    
    extent_value = extent_value.strip()
    
    # Check if it's in SQ.Yd format (e.g., "190SQ.Yd" or "190 SQ.Yd" or "190.5SQ.Yd")
    sq_yd_match = re.search(r"([\d.]+)\s*SQ\.?Yd\.?", extent_value, re.IGNORECASE)
    
    if sq_yd_match:
        try:
            sq_yd = float(sq_yd_match.group(1))
            sq_ft = sq_yd * 9  # Direct conversion: 1 Square Yard = 9 Square Feet
            # Format to 2 decimal places - REMOVED UNIT
            return f"{sq_ft:.2f}"
        except ValueError:
            return ""
    
    return ""

def convert_built_to_sq_ft(built_value: str) -> str:
    """Convert built-up area value from SQ.Ft to SQ.Ft (no conversion needed, just formatting)"""
    if pd.isna(built_value) or not isinstance(built_value, str) or not built_value:
        return ""
    
    built_value = built_value.strip()
    
    # Check if it's in SQ.Ft format - handles "50SQ. FT", "50 SQ.FT", "50SQ.Ft", "50 SQ. FT", etc.
    sq_ft_match = re.search(r"([\d.]+)\s*SQ\.?\s*FT\.?", built_value, re.IGNORECASE)
    
    if sq_ft_match:
        try:
            sq_ft = float(sq_ft_match.group(1))
            # Format to 2 decimal places - REMOVED UNIT
            return f"{sq_ft:.2f}"
        except ValueError:
            return ""
    
    return ""

def extract_dates(date_text: str) -> dict:
    """Extract Registration, Execution, and Presentation dates"""
    out = {"Registration Date": "", "Execution Date": "", "Presentation Date": ""}
    if pd.isna(date_text) or not isinstance(date_text, str):
        return out
    
    # Pattern to match (R) date, (E) date, (P) date
    r_match = re.search(r"\(R\)\s*(\d{1,2}-\d{1,2}-\d{4})", date_text, re.IGNORECASE)
    e_match = re.search(r"\(E\)\s*(\d{1,2}-\d{1,2}-\d{4})", date_text, re.IGNORECASE)
    p_match = re.search(r"\(P\)\s*(\d{1,2}-\d{1,2}-\d{4})", date_text, re.IGNORECASE)
    
    if r_match:
        out["Registration Date"] = r_match.group(1)
    if e_match:
        out["Execution Date"] = e_match.group(1)
    if p_match:
        out["Presentation Date"] = p_match.group(1)
    
    return out

def extract_document_info(doc_text: str) -> dict:
    """Extract Document type code, Document Type, Market Value, Consideration Value"""
    out = {
        "Document type code": "", 
        "Document Type": "", 
        "Market Value": "", 
        "Consideration Value": ""
    }
    if pd.isna(doc_text) or not isinstance(doc_text, str):
        return out
    
    # Extract document type code (first 4 digits)
    code_match = re.search(r"^(\d{4})", doc_text.strip())
    if code_match:
        out["Document type code"] = code_match.group(1)
    
    # Extract document type (between code and Mkt.Value)
    doc_type_match = re.search(r"^\d{4}\s+(.+?)(?:\s+Mkt\.Value:|$)", doc_text, re.IGNORECASE)
    if doc_type_match:
        out["Document Type"] = clean_spaces(doc_type_match.group(1))
    
    # Extract Market Value
    mkt_match = re.search(r"Mkt\.Value:\s*(?:Rs\.?)?\s*([0-9,]+)", doc_text, re.IGNORECASE)
    if mkt_match:
        out["Market Value"] = mkt_match.group(1).replace(",", "")
    
    # Extract Consideration Value
    cons_match = re.search(r"Cons\.Value:\s*(?:Rs\.?)?\s*([0-9,]+)", doc_text, re.IGNORECASE)
    if cons_match:
        out["Consideration Value"] = cons_match.group(1).replace(",", "")
    
    return out

def extract_parties(parties_text: str) -> dict:
    """Extract Seller and Buyer from party information - supports multiple sellers and buyers"""
    out = {"Seller": "", "Buyer": ""}
    if pd.isna(parties_text) or not isinstance(parties_text, str):
        return out
    
    # Clean the text first
    parties_text = clean_spaces(parties_text)
    
    # Remove duplicate content (if the text is repeated)
    text_length = len(parties_text)
    half_length = text_length // 2
    
    if text_length > 20 and parties_text[:half_length] == parties_text[half_length:]:
        parties_text = parties_text[:half_length]
    
    sellers = []
    buyers = []
    seen_sellers = set()
    seen_buyers = set()
    
    # FIRST: Split by numbered entries (1., 2., 3., etc.) - THIS IS CRITICAL
    # This regex splits on spaces followed by a number and dot
    entries = re.split(r'\s+(?=\d+\.)', parties_text)
    
    # If splitting didn't work well, try alternative split
    if len(entries) <= 1:
        # Find all numbered entries
        entries = re.findall(r'\d+\.[^.]*(?:\([^)]+\)[^.]*)*', parties_text)
    
    # Process each numbered entry separately
    for entry in entries:
        entry = entry.strip()
        if not entry:
            continue
        
        # Extract the number and the rest
        number_match = re.match(r'(\d+)\.\s*(.*)', entry)
        if number_match:
            number, content = number_match.groups()
        else:
            content = entry
        
        # Check for party type in this entry
        found_type = None
        for party_type in ALL_PARTY_TYPES:
            type_pattern = rf'\(({party_type})\)'
            type_match = re.search(type_pattern, content, re.IGNORECASE)
            if type_match:
                found_type = party_type.upper()
                # Remove the party type tag from content
                content = re.sub(type_pattern, '', content, flags=re.IGNORECASE).strip()
                break
        
        if not found_type:
            continue
        
        # Clean up the name
        name = content.strip()
        
        # Remove any trailing number patterns
        name = re.sub(r'\s+\d+\.\s*$', '', name)
        name = re.sub(r'^\s+|\s+$', '', name)
        
        # Skip if name is empty
        if not name:
            continue
        
        # Create a normalized version for duplicate checking
        normalized_name = re.sub(r'\s+', '', name.upper())
        normalized_name = re.sub(r'[^\w\s]', '', normalized_name)
        
        # Add to appropriate list based on party type
        if found_type in SELLER_TYPES:
            if normalized_name and normalized_name not in seen_sellers:
                sellers.append(name)
                seen_sellers.add(normalized_name)
        elif found_type in BUYER_TYPES:
            if normalized_name and normalized_name not in seen_buyers:
                buyers.append(name)
                seen_buyers.add(normalized_name)
    
    # Join with newlines
    out["Seller"] = "\n".join(sellers) if sellers else ""
    out["Buyer"] = "\n".join(buyers) if buyers else ""
    
    return out

def segment_fields(text: str) -> dict:
    """Extract property description fields from text"""
    out = {k: "" for k in ["VILL/COL", "W-B", "SURVEY", "PLOT", "HOUSE", "APARTMENT", "BLOCK", "FLAT", "EXTENT", "BUILT", "Boundires"]}
    if pd.isna(text):
        return out

    t = normalize_basic(text)
    
    # ---- Fix 1: Ensure proper spacing before "Boundires:" ----
    # Add space before "Boundires:" if it's attached to other text
    t = re.sub(r"([^ ])Boundires:", r"\1 Boundires:", t, flags=re.IGNORECASE)
    t = re.sub(r"([^ ])bound\w*:", r"\1 bound:", t, flags=re.IGNORECASE)
    
    # ---- 1) Boundaries: regex slice (most reliable) ----
    bb = BOUND_START_RE.search(t)
    if bb:
        # Get everything after the boundary marker
        remaining_text = t[bb.end():]
        # Find where the next label might start (to capture complete boundaries)
        next_label_pos = len(remaining_text)
        
        # Look for any of the other labels that might come after boundaries
        for label in ["VILL/COL:", "W-B:", "SURVEY:", "PLOT:", "HOUSE:", "APARTMENT:", "BLOCK:", "FLAT:", "EXTENT:", "BUILT:"]:
            pos = remaining_text.upper().find(label.upper())
            if 0 < pos < next_label_pos:
                next_label_pos = pos
        
        out["Boundires"] = clean_spaces(remaining_text[:next_label_pos])
        # Remove boundaries part from text so it doesn't interfere with other parsing
        t_main = t[:bb.start()].strip()
    else:
        # ---- Alternative: Try to find boundaries at the end of string ----
        # Look for boundary pattern at the end (common pattern with [N], [S], etc.)
        bound_pattern_at_end = re.search(r"(?i)(bound\w*\s*:.*?)(?:\[[NSEW]\].*?)+$", t)
        if bound_pattern_at_end:
            out["Boundires"] = clean_spaces(bound_pattern_at_end.group(1).split(":", 1)[1])
            t_main = t[:bound_pattern_at_end.start()].strip()
        else:
            t_main = t

    # ---- NEW: Handle VILL/COL when it's at the beginning without label ----
    # Check if text starts with something that's not a known label (like "VENKATGIRI-1")
    if t_main and not t_main.upper().startswith(("VILL/COL:", "W-B:", "SURVEY:", "PLOT:", "HOUSE:", "APARTMENT:", "BLOCK:", "FLAT:", "EXTENT:", "BUILT:")):
        # Extract the first part until we hit a known label
        first_part_match = re.match(r"^([^:]+?)\s+(?=W-B:|SURVEY:|PLOT:|HOUSE:|APARTMENT:|BLOCK:|FLAT:|EXTENT:|BUILT:|Boundires:)", t_main, re.IGNORECASE)
        if first_part_match:
            out["VILL/COL"] = clean_spaces(first_part_match.group(1))
            # Remove the extracted part from t_main for further processing
            t_main = t_main[len(first_part_match.group(1)):].strip()

    # ---- 2) Parse other fields by simple label splits ----
    # Ensure labels have ":" (only for known labels)
    # Be careful with HOUSE to not match "/HOUSE SITE" in VILL/COL
    for k in ["VILL/COL", "W-B", "SURVEY", "PLOT", "APARTMENT", "BLOCK", "FLAT", "EXTENT", "BUILT"]:
        t_main = re.sub(rf"(?i)\b{k}\b\s*(?!:)", f"{k}:", t_main)
    
    # Handle HOUSE separately with a more precise pattern
    # Only add colon if HOUSE is at word boundary and not part of VILL/COL
    t_main = re.sub(rf"(?i)(?<!/)\bHOUSE\b\s*(?!:)", "HOUSE:", t_main)

    # Clean up any double colons
    t_main = re.sub(r"::+", ":", t_main)

    def grab(label, s):
        # Improved to better detect field boundaries
        # Special handling for HOUSE to avoid matching in VILL/COL
        if label == "HOUSE":
            # More precise pattern for HOUSE
            m = re.search(rf"(?i)(?<!/)\b{re.escape(label)}\s*:\s*(.*?)(?=\s+(?:VILL/COL|W-B|SURVEY|PLOT|APARTMENT|BLOCK|FLAT|EXTENT|BUILT|Boundires)\s*:|$)", s)
        else:
            m = re.search(rf"(?i)\b{re.escape(label)}\s*:\s*(.*?)(?=\s+(?:VILL/COL|W-B|SURVEY|PLOT|HOUSE|APARTMENT|BLOCK|FLAT|EXTENT|BUILT|Boundires)\s*:|$)", s)
        
        if m:
            value = m.group(1)
            # Additional cleanup: remove any trailing text that might contain next field's label
            value = re.sub(r'\s+(?:VILL/COL|W-B|SURVEY|PLOT|HOUSE|APARTMENT|BLOCK|FLAT|EXTENT|BUILT|Boundires)\s*:.*$', '', value, flags=re.IGNORECASE)
            return clean_spaces(value)
        return ""

    # Only grab VILL/COL if we didn't already extract it from the beginning
    if not out["VILL/COL"]:
        out["VILL/COL"] = grab("VILL/COL", t_main)
    
    out["W-B"]      = grab("W-B", t_main)
    out["SURVEY"]   = grab("SURVEY", t_main)
    out["PLOT"]     = grab("PLOT", t_main)
    out["HOUSE"]    = grab("HOUSE", t_main)
    out["APARTMENT"] = grab("APARTMENT", t_main)
    out["BLOCK"]    = grab("BLOCK", t_main)
    out["FLAT"]     = grab("FLAT", t_main)
    out["EXTENT"]   = grab("EXTENT", t_main)
    out["BUILT"]    = grab("BUILT", t_main)

    # ---- 3) Clean EXTENT/BUILT ----
    # Handle the case where boundaries text might have leaked into EXTENT
    if out["EXTENT"] and any(bound_word in out["EXTENT"].upper() for bound_word in ["BOUND", "[N]", "[S]", "[E]", "[W]"]):
        # Split on boundary markers
        for bound_marker in [" bound", " Bound", " BOUND", "[N]", "[S]", "[E]", "[W]"]:
            if bound_marker in out["EXTENT"]:
                out["EXTENT"] = out["EXTENT"].split(bound_marker)[0].strip()
                break
    
    out["EXTENT"] = pick_unit_token(out["EXTENT"], "EXTENT")
    out["BUILT"]  = pick_unit_token(out["BUILT"], "BUILT")

    # ---- Final check: If boundaries still empty but we see boundary patterns in main text ----
    if not out["Boundires"]:
        # Look for boundary pattern anywhere in original text
        bound_match = re.search(r"(?i)(?:bound\w*\s*:|\b(?:north|south|east|west|n|s|e|w)[\s:]*).*?(?:\[[NSEW]\].*?)+", t)
        if bound_match:
            # Extract just the boundary description part
            bound_text = bound_match.group(0)
            if ":" in bound_text:
                out["Boundires"] = clean_spaces(bound_text.split(":", 1)[1])
            else:
                out["Boundires"] = clean_spaces(bound_text)

    return out

# ===== CLASSIFICATION FUNCTION FOR TRANSACTION TYPE =====
def classify_transaction(doc_type):
    sales_types = [
        "Sale Deed",
        "AGREEMENT OF SALE CUM GPA",
        "Sale Agreement Without Possess",
        "Sale Agreement With Possession",
        "CONVEYANCE FOR CONSIDERATION",
        "Sale deed executed by A.P.Hous",
        "Sale Deeds executed by Courts",
        "Sale Certificate",
        "Assignment deed",
        "RECONVEYANCE DEED EXECUTED BY",
        "Sale of life interest",
        "Sale deed executed by or infav",
        "Sale deed executed by Society",
        "Sale deed in favour of State o"
    ]

    lease_types = [
        "Lease Deed",
        "Lease in favour of State/Centr",
        "Lease(others)",
        "Surrender of Lease",
        "Transfer of Lease"
    ]
    
    if pd.isna(doc_type):
        return "Others"

    doc_type = str(doc_type).strip()

    if doc_type in sales_types:
        return "Sale"
    elif doc_type in lease_types:
        return "Lease"
    else:
        return "Other"

# ===== ENHANCED FUNCTION FOR PROPERTY TYPE CLASSIFICATION =====
# ===== ENHANCED FUNCTION FOR PROPERTY TYPE CLASSIFICATION =====
# ===== ENHANCED FUNCTION FOR PROPERTY TYPE CLASSIFICATION =====
import re
import pandas as pd


# ===== PROCESS THE DATAFRAME =====
print("Starting processing of df_clean...")

# Make a copy to avoid modifying the original
df_processed = df_clean.copy()

src_col = "Description of property"

# Extract property description fields
print("Extracting property description fields...")
parsed = df_processed[src_col].apply(segment_fields).apply(pd.Series)
df_processed = pd.concat([df_processed, parsed], axis=1)

# Add EXTENT in SqFt column (direct conversion from SQ.Yd to SQ.Ft)
print("Converting EXTENT to SqFt...")
df_processed["EXTENT in SqFt"] = df_processed["EXTENT"].apply(convert_extent_to_sq_ft)

# Add BUILT in SqFt column (no conversion, just formatting)
print("Converting BUILT to SqFt...")
df_processed["BUILT in SqFt"] = df_processed["BUILT"].apply(convert_built_to_sq_ft)

# Extract dates from Reg.Date Exe.Date Pres.Date column
if "Reg.Date Exe.Date Pres.Date" in df_processed.columns:
    print("Extracting dates...")
    dates_parsed = df_processed["Reg.Date Exe.Date Pres.Date"].apply(extract_dates).apply(pd.Series)
    # Insert date columns after the original date column
    date_col_idx = df_processed.columns.get_loc("Reg.Date Exe.Date Pres.Date") + 1
    for i, col in enumerate(["Registration Date", "Execution Date", "Presentation Date"]):
        df_processed.insert(date_col_idx + i, col, dates_parsed[col])

# Extract document info from Nature & Mkt.Value Con. Value column
if "Nature & Mkt.Value Con. Value" in df_processed.columns:
    print("Extracting document information...")
    doc_parsed = df_processed["Nature & Mkt.Value Con. Value"].apply(extract_document_info).apply(pd.Series)
    # Insert document columns after the original document column
    doc_col_idx = df_processed.columns.get_loc("Nature & Mkt.Value Con. Value") + 1
    for i, col in enumerate(["Document type code", "Document Type", "Market Value", "Consideration Value"]):
        df_processed.insert(doc_col_idx + i, col, doc_parsed[col])

# Extract seller and buyer from Name of Parties Executant(EX) & Claimants(CL) column
if "Name of Parties Executant(EX) & Claimants(CL)" in df_processed.columns:
    print("Extracting seller and buyer information...")
    parties_parsed = df_processed["Name of Parties Executant(EX) & Claimants(CL)"].apply(extract_parties).apply(pd.Series)
    # Insert party columns after the original party column
    party_col_idx = df_processed.columns.get_loc("Name of Parties Executant(EX) & Claimants(CL)") + 1
    for i, col in enumerate(["Seller", "Buyer"]):
        df_processed.insert(party_col_idx + i, col, parties_parsed[col])

# ===== ADD TRANSACTION TYPE COLUMN AFTER DOCUMENT TYPE =====
# Check if Document Type column exists (it should from the extraction above)
if "Document Type" in df_processed.columns:
    print("Adding Transaction Type column...")
    # Create transaction type values
    transaction_values = df_processed["Document Type"].apply(classify_transaction)
    
    # Insert Transaction Type column after Document Type
    doc_type_idx = df_processed.columns.get_loc("Document Type")
    df_processed.insert(doc_type_idx + 1, "Transaction Type", transaction_values)
    
    
    # ===== PRINT TRANSACTION TYPE SUMMARY =====
    print("\n" + "="*60)
    print("TRANSACTION TYPE SUMMARY")
    print("="*60)
    transaction_summary = df_processed["Transaction Type"].value_counts()
    sales_count = transaction_summary.get("Sales", 0)
    lease_count = transaction_summary.get("Lease", 0)
    others_count = transaction_summary.get("Others", 0)
    total_count = len(df_processed)
    
    # Print in the requested format
    print(f"Sales\tLease\tOthers\tTotal")
    print(f"{sales_count}\t{lease_count}\t{others_count}\t{total_count}")
    print("="*60)

# ---- Debug: Show rows where Boundaries is still empty ----
empty_boundaries = df_processed[df_processed["Boundires"] == ""]
if not empty_boundaries.empty:
    print(f"Found {len(empty_boundaries)} rows with empty boundaries:")
    for idx, row in empty_boundaries.head(10).iterrows():
        print(f"\nRow {idx}:")
        print(f"Original text: {row[src_col]}")
        print("-" * 50)

# Debug: Show sample of BUILT and BUILT in SqFt to verify conversion
print("\n--- BUILT Conversion Sample (first 10 rows) ---")
sample_rows = df_processed[["BUILT", "BUILT in SqFt"]].head(10)
print(sample_rows.to_string())

# Debug: Show sample of EXTENT and EXTENT in SqFt to verify conversion
print("\n--- EXTENT Conversion Sample (first 10 rows) ---")
sample_rows = df_processed[["EXTENT", "EXTENT in SqFt"]].head(10)
print(sample_rows.to_string())

# Debug: Show sample of parties extraction with new mappings
print("\n--- Parties Extraction Sample with MR/ME, DR/DE, RR/RE Mappings (first 20 rows) ---")
if "Name of Parties Executant(EX) & Claimants(CL)" in df_processed.columns:
    # Get rows that might contain the new party types
    sample_df = df_processed[["Name of Parties Executant(EX) & Claimants(CL)", "Seller", "Buyer"]].head(20)
    
    # Also check specifically for rows with MR, ME, DR, DE, RR, RE
    print("\nRows with new party types (MR/ME, DR/DE, RR/RE, PL/AY, LR/LE):")
    for idx, row in df_processed.head(50).iterrows():
        text = str(row["Name of Parties Executant(EX) & Claimants(CL)"])
        if any(x in text for x in ['(MR)', '(ME)', '(DR)', '(DE)', '(RR)', '(RE)', '(PL)', '(AY)', '(LR)', '(LE)', '(FP)', '(SP)']):
            print(f"\nRow {idx}:")
            print(f"Original: {text[:100]}..." if len(text) > 100 else f"Original: {text}")
            print(f"Seller:\n{row['Seller']}")
            print(f"Buyer:\n{row['Buyer']}")
            print("-" * 40)
    
    print("\nFirst 20 rows sample:")
    print(sample_df.to_string())

print(f"\nProcessing complete! df_processed now has {len(df_processed.columns)} columns and {len(df_processed)} rows.")
print("You can continue working with df_processed for further analysis.")

Starting processing of df_clean...
Extracting property description fields...
Converting EXTENT to SqFt...
Converting BUILT to SqFt...
Extracting dates...
Extracting document information...
Extracting seller and buyer information...
Adding Transaction Type column...
Adding Property Type column...

TRANSACTION TYPE SUMMARY
Sales	Lease	Others	Total
24408	2175	35977	62560

PROPERTY TYPE SUMMARY
House: 34455
Flat: 23254
Shop: 2966
Land: 1293
Others: 592
Found 22 rows with empty boundaries:

Row 1374:
Original text: VILL/COL: Hyderabad W-B: 0-0
--------------------------------------------------

Row 2749:
Original text: VILL/COL: Hyderabad W-B: 0-0
--------------------------------------------------

Row 3025:
Original text: VILL/COL: Hyderabad W-B: 0-0
--------------------------------------------------

Row 5449:
Original text: W-B: 0-0 BUILT: 1SQ. FT
--------------------------------------------------

Row 9967:
Original text: VILL/COL: Hyderabad/CHATRINAKA-1 W-B: 18-3
----------------------

### Proprty type filled with flat Where FLAT Column Contain Digit

In [ ]:
# Check each value in the actual column and put "FLAT" if it's a whole number
df_processed['Property_type'] = df_processed['FLAT'].apply(
    lambda x: 'FLAT' if str(x).strip().isdigit() else ''
)

# View the result
print(df_processed[['FLAT', 'Property_type']])

In [ ]:
flats_only = df_processed[df_processed['Property_type'] == 'FLAT']
print(flats_only)

In [2]:
processed_df=pd.read_excel(r"D:\Nilesh\Telangana Processing\2024\Yadadri Bhuvanagiri\Yadadri_Bhuvanagiri_Cleaned&Extracted.xlsx")
processed_df.head(2)

,S.No.,Description of property,Reg.Date Exe.Date Pres.Date,Registration Date,Execution Date,Presentation Date,Nature & Mkt.Value Con. Value,Document type code,Document Type,Transaction Type,...,HOUSE,APARTMENT,BLOCK,FLAT,EXTENT,BUILT,Boundires,EXTENT in SqFt,BUILT in SqFt,Property_type
0,1.0,VILL/COL: MARYALA/MARYALA VILLAGE W-B: 0-0 HOU...,(R) 31-12-2024 (E) 31-12-2024 (P) 31-12-2024,31-12-2024,31-12-2024,31-12-2024,0208 Deposit of Title Deeds Mkt.Value:Rs. 9171...,208.0,Deposit of Title Deeds,Others,...,2/87,NaN,NaN,NaN,311SQ.Yds,240SQ. FT,[N]: HOUSE OF TUNIKI MUTHYAM GOUD [S] HOUSE OF...,2799.0,240.0,NaN
1,1.0,VILL/COL: BHUVANAGIRI/JALEELPURA W-B: 3-5 HOUS...,(R) 31-12-2024 (E) 31-12-2024 (P) 31-12-2024,31-12-2024,31-12-2024,31-12-2024,0801 Rectification Deed Mkt.Value:Rs. 149100 C...,801.0,Rectification Deed,Others,...,3-5-34,NaN,NaN,NaN,42SQ.Yds,200SQ. FT,[N]: WAY [S] HOUSE OF MOHD TAHER [E]: PUBLIC W...,378.0,200.0,NaN


### Property Type Using Regex

In [4]:
import re
import pandas as pd
from tqdm import tqdm

def extract_core_property_text(description_text):
    """
    Extract only the core property information, removing ALL boundaries data
    """
    if not isinstance(description_text, str):
        return ""
    
    # Work with original case for now, will convert to uppercase later if needed
    text = description_text
    
    # =========================
    # STRATEGY 1: Split at boundaries marker and take first part
    # =========================
    boundaries_markers = [
        r'\bBoundires\s*:',
        r'\bBoundaries\s*:',
        r'\bBoundry\s*:',
        r'\bBounds\s*:',
        r'\bBoundaries\s*\[',
        r'\bBoundires\s*\[',
    ]
    
    core_text = text
    for marker in boundaries_markers:
        parts = re.split(marker, text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) > 1:
            core_text = parts[0]
            break
    
    # =========================
    # STRATEGY 2: Remove everything after bracket patterns like [N]:, [S]:, etc.
    # =========================
    # Remove any [N]:, [S]:, [E]:, [W]: patterns and everything after them
    core_text = re.sub(r'\[[NSWE]\]\s*:.*$', '', core_text, flags=re.IGNORECASE)
    
    # Remove standalone brackets with content
    core_text = re.sub(r'\[[^\]]+\]', '', core_text)
    
    # =========================
    # STRATEGY 3: Remove lines containing boundary keywords
    # =========================
    boundary_keywords = [
        r'Boundires', r'Boundaries', r'Boundry', r'Bounds',
        r'\[N\]', r'\[S\]', r'\[E\]', r'\[W\]',
        r'Gramapanchayath', r'Gram Panchayat', r'Municipal Office',
        r'Road\s*$', r'Wide Road', r'House Of', r'Land Of',
        r'OFFICE OF THE GRAMPANCHAYATH', r'MUNICIPAL OFFICE'
    ]
    
    for keyword in boundary_keywords:
        core_text = re.sub(rf'.*{keyword}.*(\n|$)', '', core_text, flags=re.IGNORECASE)
    
    # =========================
    # STRATEGY 4: Keep only the first part before any boundary indicators
    # =========================
    match = re.search(r'(\[N\]\s*:|\[S\]\s*:|\[E\]\s*:|\[W\]\s*:)', core_text, re.IGNORECASE)
    if match:
        core_text = core_text[:match.start()]
    
    # Clean up extra spaces and normalize
    core_text = re.sub(r'\s+', ' ', core_text).strip()
    
    return core_text


def classify_property_type_regex(description_text):
    """
    Classify property type using regex patterns
    STRICTLY avoids ALL boundaries data
    Returns one of: Flat, Shop, Office, Parking, Plot, House, Others
    """
    if not isinstance(description_text, str) or not description_text.strip():
        return "Others"
    
    # =========================
    # FIRST: Extract only core property text (remove ALL boundaries)
    # =========================
    core_text = extract_core_property_text(description_text)
    
    # If core text is empty, use original but with boundaries removed
    if not core_text:
        # Fallback: remove everything after "Boundires:" or similar
        boundaries_pattern = r'(?:Boundires|Boundaries|Boundry|Bounds)\s*:.*$'
        core_text = re.sub(boundaries_pattern, '', description_text, flags=re.IGNORECASE | re.DOTALL)
        core_text = re.sub(r'\[[^\]]*\]', '', core_text)
        core_text = re.sub(r'\s+', ' ', core_text).strip()
    
    # If still empty, return Others
    if not core_text:
        return "Others"
    
    # Convert to uppercase for case-insensitive matching
    text_upper = core_text.upper()
    
    # =========================
    # CLASSIFICATION ON CORE TEXT ONLY
    # =========================
    
    # =========================
    # PATTERN 1: SHOP (MUST COME FIRST - Check for SHOP in flat/apartment names)
    # =========================
    shop_patterns = [
        r'\bFLAT\s*:\s*SHOP\d+\b',                           # FLAT: SHOP408, FLAT: SHOP14
        r'\bFLAT\s*:\s*SHOP[A-Z0-9]+\b',                     # FLAT: SHOPG3A, FLAT: SHOPG-5
        r'\bFLAT\s*:\s*\d+SHOP\b',                           # FLAT: 408SHOP
        r'\bFLAT\s*:\s*SHOP\s*[A-Z0-9-]+\b',                 # FLAT: SHOP G-5
        r'\bAPARTMENT\s*:.*\bFLAT\s*:\s*SHOP\d+\b',          # APARTMENT: X FLAT: SHOP408
        r'\bAPARTMENT\s*:.*\bFLAT\s*:\s*SHOP[A-Z0-9]+\b',    # APARTMENT: X FLAT: SHOPG3A
        r'\bSHOP\s+(?:NO|NUMBER)\s*:?\s*\d+',                # SHOP NO 123
        r'\bSHOP\s*:\s*\d+',                                 # SHOP: 123
        r'\bSHOP\s*:\s*[A-Z0-9-]+',                          # SHOP: identifier
        r'\bCOMMERCIAL\s+SHOP\b',                            # Commercial shop
        r'\bSHOP\s+NO\s*\d+\b',                              # SHOP NO 8
        r'\bSHOP\s*:\s*[A-Z]+\b',                            # SHOP: X
        r'\bRETAIL\s+SHOP\b',                                # Retail shop
        r'\bSHOP\b(?!.*\b(?:HOUSE|FLAT)\b)',                 # Shop without house/flat context
    ]
    
    for pattern in shop_patterns:
        if re.search(pattern, text_upper):
            return "Shop"
    
    # =========================
    # PATTERN 2: OFFICE (Enhanced for all office variations)
    # =========================
    office_patterns = [
        r'\bFLAT\s*:\s*OFFICE\b',                            # FLAT: OFFICE
        r'\bFLAT\s*:\s*OFC\s+[IVX]+\b',                     # FLAT: OFC I, OFC II, OFC III
        r'\bFLAT\s*:\s*OFF\s+[0-9A-Z-]+\b',                 # FLAT: OFF 4-C
        r'\bFLAT\s*:\s*\d+/OFF\b',                           # FLAT: 403/OFF
        r'\bFLAT\s*:\s*OFFICE\s+SPACE\b',                    # FLAT: OFFICE SPACE
        r'\bFLAT\s*:\s*OFC\b',                               # FLAT: OFC
        r'\bAPARTMENT\s*:.*\bFLAT\s*:\s*OFFICE\b',           # APARTMENT: X FLAT: OFFICE
        r'\bAPARTMENT\s*:.*\bFLAT\s*:\s*OFC\b',              # APARTMENT: X FLAT: OFC
        r'\bAPARTMENT\s*:.*\bFLAT\s*:\s*OFF\b',              # APARTMENT: X FLAT: OFF
        r'\bOFFICE\s+(?:NO|NUMBER)\s*:?\s*\d+',              # OFFICE NO 123
        r'\bOFFICE\s*:\s*\d+',                               # OFFICE: 123
        r'\bOFFICE\s*:\s*[A-Z0-9-]+',                       # OFFICE: identifier
        r'\bOFFICE\s+SPACE\b',                               # Office space
        r'\bOFFICE\s+SUITE\b',                               # Office suite
        r'\bCOMMERCIAL\s+OFFICE\b',                          # Commercial office
        r'\bCORPORATE\s+OFFICE\b',                           # Corporate office
        r'\bOFFICE\b(?!.*\b(?:HOUSE)\b)',                    # Office not near house
    ]
    
    for pattern in office_patterns:
        if re.search(pattern, text_upper):
            # Double-check it's not from boundaries
            if not re.search(r'(?:MUNICIPAL|GRAMAPANCHAYATH|GRAM\s+PANCHAYAT|GOVERNMENT)', text_upper):
                return "Office"
    
    # =========================
    # PATTERN 3: FLAT/APARTMENT (Now after SHOP and OFFICE)
    # =========================
    flat_patterns = [
        r'\bAPARTMENT\s*:.*\bFLAT\s+(?:NO|NUMBER)?\s*:?\s*\d+',  # APARTMENT: X FLAT: 123
        r'\bAPARTMENT\s*:.*\bFLAT\s+\d+',                         # APARTMENT: X FLAT 123
        r'\bAPARTMENT\s*:.*\bFLAT\s*:\s*[A-Z0-9-]+',              # APARTMENT: X FLAT: identifier
        r'\bAPARTMENT\s*:\s*[A-Z0-9\s]+\s+FLAT\s*:\s*\d+',        # APARTMENT: NAME FLAT: number
        r'\bAPARTMENT\s*:\s*[A-Z0-9\s]+\s+FLAT\s+\d+',            # APARTMENT: NAME FLAT number
        r'\bFLAT\s+(?:NO|NUMBER)\s*:?\s*\d+',                      # FLAT NO 123
        r'\bFLAT\s*:\s*\d+',                                      # FLAT: 123
        r'\bFLAT\s*:\s*[A-Z0-9-]+',                               # FLAT: identifier
        r'\bFLAT\s+[A-Z]\d+\b',                                   # FLAT A11
        r'\bFLAT\s+NO\s*\d+\b',                                   # FLAT NO 403
        r'\bAPARTMENT\s*:\s*[A-Z0-9\s]+',                         # APARTMENT: NAME
        r'\bMIG-\d+\b',                                          # MIG housing scheme
        r'\bLIG-\d+\b',                                          # LIG housing scheme
        r'\bSCHEDULE\s+[A-Z]\d+\b',                              # Schedule A11
        r'\bFLAT\b.*\bFLOOR\b',                                  # Flat with floor
        r'\bFLOOR\b.*\bFLAT\b',                                  # Floor with flat
    ]
    
    for pattern in flat_patterns:
        if re.search(pattern, text_upper):
            return "Flat"
    
    # =========================
    # PATTERN 4: HOUSE
    # =========================
    house_patterns = [
        r'\bHOUSE\s+(?:NO|NUMBER)\s*:?\s*\d+(?:[-\/\d]*)?',  # HOUSE: 1/88, HOUSE NO 123
        r'\bHOUSE\s*:\s*\d+(?:[-\/\d]*)?',                    # HOUSE: 123
        r'\bHOUSE\s*:\s*[A-Z0-9-]+',                         # HOUSE: identifier
        r'\bHOUSE\s+[A-Z]\d+\b',                              # HOUSE A11
        r'\bRESIDENTIAL\s+HOUSE\b',
        r'\bDETACHED\s+HOUSE\b',
        r'\bHOUSE\b',                                         # Any house mention
    ]
    
    for pattern in house_patterns:
        if re.search(pattern, text_upper):
            return "House"
    
    # =========================
    # PATTERN 5: PARKING
    # =========================
    parking_patterns = [
        r'\bPARKING\s+(?:SLOT|SPACE|AREA)\b',
        r'\bPARKING\s+NO\b',
        r'\bDRIVE\s+WAY\b',
        r'\bCAR\s+PARK\b',
        r'\bPARKING\s*:\s*X\b',
        r'\bPARKING\b',
    ]
    
    for pattern in parking_patterns:
        if re.search(pattern, text_upper):
            return "Parking"
    
    # =========================
    # PATTERN 6: PLOT/LAND
    # =========================
    plot_patterns = [
        r'\bPLOT\s+(?:NO|NUMBER)\s*:?\s*\d+',
        r'\bPLOT\s*:\s*\d+',
        r'\bPLOT\s*:\s*[A-Z0-9-]+',
        r'\bPLOT\s+[A-Z]\d+\b',
        r'\bSURVEY\s*(?:NO|NUMBER)\s*:?\s*\d+',
        r'\bSY\s+NO\s*\d+\b',
        r'\bGRAMAKANTAM\b',                                  # Village land
        r'\bLAND\b(?!.*\b(?:HOUSE|FLAT)\b)',
        r'\bOPEN\s+PLOT\b',
        r'\bRESIDENTIAL\s+PLOT\b',
        r'\bSITE\s+NO\s*\d+\b',
        r'\bVACANT\s+LAND\b',
        r'\bAGRICULTURAL\s+LAND\b',
    ]
    
    for pattern in plot_patterns:
        if re.search(pattern, text_upper):
            return "Plot"
    
    # =========================
    # FALLBACK: Check for area-based classification
    # =========================
    extent_match = re.search(r'\bEXTENT\s*:\s*(\d+(?:\.\d+)?)\s*(SQ\.YDS|SQ\.FT|SQ\.M|SQ\.YARD|SQUARE\s+(?:YARDS|FEET|METERS))', text_upper)
    built_match = re.search(r'\bBUILT\s*:\s*(\d+(?:\.\d+)?)\s*(SQ\.FT|SQ\.M|SQUARE\s+(?:FEET|METERS))', text_upper)
    
    if extent_match and built_match:
        extent_value = float(extent_match.group(1))
        built_value = float(built_match.group(1))
        extent_unit = extent_match.group(2)
        
        # Convert to consistent unit
        if 'YDS' in extent_unit or 'YARD' in extent_unit:
            extent_sqft = extent_value * 9
        else:
            extent_sqft = extent_value
        
        # Check if built area is substantial (likely a building)
        if built_value > 500:
            # Look for shop keywords first
            if re.search(r'\bSHOP\b', text_upper):
                return "Shop"
            # Look for office keywords
            elif re.search(r'\bOFFICE\b', text_upper) or re.search(r'\bOFC\b', text_upper):
                return "Office"
            # Look for residential keywords
            elif re.search(r'\bHOUSE\b', text_upper) or re.search(r'\bRESIDENTIAL\b', text_upper):
                return "House"
            elif re.search(r'\bFLAT\b', text_upper) or re.search(r'\bAPARTMENT\b', text_upper):
                return "Flat"
            else:
                return "House"  # Default to house for residential properties
    
    # =========================
    # FINAL FALLBACK: Check for property type indicators
    # =========================
    if re.search(r'\bSHOP\b', text_upper):
        return "Shop"
    elif re.search(r'\bOFFICE\b', text_upper) or re.search(r'\bOFC\b', text_upper):
        return "Office"
    elif re.search(r'\bHOUSE\b', text_upper):
        return "House"
    elif re.search(r'\bFLAT\b', text_upper) or re.search(r'\bAPARTMENT\b', text_upper):
        return "Flat"
    elif re.search(r'\bPLOT\b', text_upper) or re.search(r'\bLAND\b', text_upper) or re.search(r'\bSURVEY\b', text_upper):
        return "Plot"
    elif re.search(r'\bPARKING\b', text_upper):
        return "Parking"
    
    return "Others"


def classify_batch_regex(descriptions, show_progress=True):
    """
    Process a batch of descriptions using regex with strict boundary avoidance
    """
    results = []
    iterator = tqdm(descriptions, desc="Classifying property types") if show_progress else descriptions
    
    for desc in iterator:
        result = classify_property_type_regex(desc)
        results.append(result)
    
    return results


# =========================
# MAIN PROCESSING FUNCTION
# =========================
def process_property_types_with_regex(processed_df, property_type_col='Property Type', 
                                      description_col=None, overwrite_existing=False):
    """
    Process DataFrame to add/update property types using regex with strict boundary avoidance
    
    Parameters:
    - processed_df: DataFrame with property descriptions
    - property_type_col: Name of property type column (default: 'Property Type')
    - description_col: Name of description column (auto-detects if None)
    - overwrite_existing: If True, overwrite all; if False, only fill blank rows
    
    Returns:
    - DataFrame with updated property types
    """
    # Auto-detect description column
    if description_col is None:
        possible_desc_cols = ['Description of property', 'Description', 'Bhumapan', 
                              'property_description', 'description']
        for col in possible_desc_cols:
            if col in processed_df.columns:
                description_col = col
                break
        
        if description_col is None:
            for col in processed_df.columns:
                if 'description' in col.lower() or 'property' in col.lower():
                    description_col = col
                    break
    
    if description_col is None:
        raise ValueError(f"No description column found. Available columns: {list(processed_df.columns)}")
    
    print(f"Using description column: '{description_col}'")
    
    # Check if property type column exists
    if property_type_col not in processed_df.columns:
        processed_df[property_type_col] = None
        print(f"Created new column: '{property_type_col}'")
        overwrite_existing = True
    
    # Determine which rows to process
    if overwrite_existing:
        rows_to_process = len(processed_df)
        print(f"Overwriting all {rows_to_process} rows")
        mask_to_process = [True] * len(processed_df)
    else:
        # Only process blank rows
        blank_mask = processed_df[property_type_col].isna() | (processed_df[property_type_col].astype(str).str.strip() == '')
        rows_to_process = blank_mask.sum()
        print(f"Processing {rows_to_process} blank rows out of {len(processed_df)} total")
        mask_to_process = blank_mask
    
    if rows_to_process == 0:
        print("No rows to process")
        return processed_df
    
    # Extract descriptions for rows to process
    descriptions = processed_df.loc[mask_to_process, description_col].fillna('').tolist()
    
    # Classify using regex with strict boundary avoidance
    print("Classifying property types with strict boundary avoidance...")
    property_types = classify_batch_regex(descriptions)
    
    # Update DataFrame
    processed_df.loc[mask_to_process, property_type_col] = property_types
    
    # Show summary
    print("\n" + "="*60)
    print("PROPERTY TYPE DISTRIBUTION:")
    print("="*60)
    value_counts = processed_df[property_type_col].value_counts()
    for prop_type, count in value_counts.items():
        percentage = (count / len(processed_df)) * 100
        print(f"  {prop_type}: {count} ({percentage:.1f}%)")
    print("="*60)
    
    # Show sample of classifications
    print("\n" + "="*60)
    print("SAMPLE CLASSIFICATIONS (First 10):")
    print("="*60)
    sample_df = processed_df[processed_df[property_type_col].notna()].head(10)
    for idx, row in sample_df.iterrows():
        desc_preview = str(row[description_col])[:100] if pd.notna(row[description_col]) else "N/A"
        print(f"\n{row[property_type_col]}: {desc_preview}...")
    print("="*60)
    
    return processed_df


# =========================
# USAGE EXAMPLE - READY TO PROCESS YOUR DATAFRAME
# =========================
if __name__ == "__main__":
    # Test cases including the new office examples
    test_cases = [
        {
            "text": "VILL/COL: PUPPALGUDA/PUPPAL GUDA W-B: 0-0 SURVEY: 282/P APARTMENT: EON-HYDERABAD FLAT: OFC I EXTENT: .5SQ.Yds BUILT: 51SQ. FT Boundires: [N]: WASHROOMS [S] STAIRCASE [E]: COMMON PASSAGE [W]: COMMON PASSAGE",
            "expected": "Office"
        },
        {
            "text": "VILL/COL: NARSINGI/COMMERCIAL-3 W-B: 0-0 SURVEY: 158/P APARTMENT: JYOTHI OPTIMA FLAT: OFF 4-C EXTENT: 35.4SQ.Yds BUILT: 3000SQ. FT Boundires: [N]: OPEN TO SKY [S] CORRIDOR [E]: OPEN TO SKY [W]: OFFICE SPACE NO.4-B",
            "expected": "Office"
        },
        {
            "text": "VILL/COL: KOKAPET/COMMERCIAL-3 W-B: 0-0 SURVEY: 107/P 108/P HOUSE: . APARTMENT: LAXMI INFOBAHN, TOWER-5 FLAT: OFFICE EXTENT: 1SQ.Yds BUILT: 2500SQ. FT Boundires: [N]: OPEN TO SKY & PART OF THE TOWER 6 OFFICE SPACE [S] OPEN TO SKY [E]: OPEN TO SKY [W]: OPEN TO SKY & PART OF THE TOWER-6 OFFICE SPACE",
            "expected": "Office"
        },
        {
            "text": "VILL/COL: PUPPALGUDA/RESIDENTIAL-2 W-B: 0-0 SURVEY: 285/P APARTMENT: TOWER-2 FLAT: OFFICE EXTENT: 77.5SQ.Yds BUILT: 3876SQ. FT Boundires: [N]: NEIGHBOURS PROPERTY [S] OPEN TO SKY [E]: NEIGHBOURS PROPERTY [W]: NEIGHBOURS PROPERTY",
            "expected": "Office"
        },
        {
            "text": "VILL/COL: NARSINGI/COMMERCIAL-3 W-B: 0-0 SURVEY: 160 161 PLOT: 10 APARTMENT: RAICHANDANI BUSINESS BAY FLAT: SHOP408 EXTENT: 30SQ.Yds BUILT: 1386SQ. FT Boundires: [N]: SHOP NO.407 [S] CORRIDOR [E]: CORRIDOR [W]: OPEN TO SKY / SETBACK",
            "expected": "Shop"
        },
        {
            "text": "VILL/COL: PUPPALGUDA/RESIDENTIAL-2 W-B: 0-0 SURVEY: 285/P APARTMENT: TOWER-6 FLAT: OFFICE EXTENT: 54SQ.Yds BUILT: 2720SQ. FT Boundires: [N]: NEIGHBOURS PROPERTY [S] OPEN TO SKY [E]: NEIGHBOURS PROPERTY [W]: NEIGHBOURS PROPERTY",
            "expected": "Office"
        },
        {
            "text": "VILL/COL: SOLIPUR/RESIDENTIAL W-B: 0-0 HOUSE: 1/88 EXTENT: 136SQ.Yds BUILT: 1120SQ. FT Boundires: [N]: 12'-0\" WIDE ROAD [S] SOLIPUR MUNICIPAL OFFICE [E]: HOUSE OF ANIMONI NARSIMULU [W]: HOUSE OF S VENKATAIAH",
            "expected": "House"
        },
        {
            "text": "VILL/COL: ANANTAWARAM/ANANTAWARAM W-B: 0-0 SURVEY: 130 131 132 133 134 144 145 PLOT: 6-RAAVIPALLE APARTMENT: RAAVI PALLE 06 EXTENT: 1500SQ.Yds BUILT: 4712SQ. FT BLOCK 1Boundires: [N]: Raavi Palle, Flat No 05 [S] Raavi Palle, Flat No 07 [E]: Drive way [W]: community Farm Area",
            "expected": "Flat"
        },
        {
            "text": "SHOP NO: 123 MAIN ROAD EXTENT: 500SQ.FT BUILT: 500SQ.FT",
            "expected": "Shop"
        },
        {
            "text": "FLAT NO: 403 TOWER B EXTENT: 1200SQ.FT BUILT: 1200SQ.FT",
            "expected": "Flat"
        }
    ]
    
    print("="*60)
    print("TESTING WITH OFFICE, SHOP, AND FLAT CLASSIFICATIONS:")
    print("="*60)
    
    for i, test in enumerate(test_cases, 1):
        result = classify_property_type_regex(test["text"])
        status = "✓" if result == test["expected"] else "✗"
        print(f"\nTest {i} {status}:")
        print(f"  Expected: {test['expected']}")
        print(f"  Got: {result}")
        print(f"  Preview: {test['text'][:100]}...")
        
        # Show what the core text looks like after boundary removal
        core_text = extract_core_property_text(test["text"])
        print(f"  Core text: {core_text[:100]}...")
    
    print("\n" + "="*60)
    print("PROCESSING YOUR DATAFRAME")
    print("="*60)
    
    # Process your actual DataFrame
    # Make sure 'processed_df' is already loaded with your data
    processed_df = process_property_types_with_regex(
        processed_df,
        property_type_col='Property Type',
        overwrite_existing=False  # Set to True to overwrite all existing property types
    )
    
    # Save the updated DataFrame
    output_path = r"D:\Nilesh\Telangana Processing\2024\Yadadri Bhuvanagiri\Yadadri_Bhuvangiri_With_Property_Type.xlsx"
    processed_df.to_excel(output_path, index=False)
    print(f"\n✅ Saved to: {output_path}")
    
    # Verify results
    blank_after = processed_df['Property Type'].isna().sum()
    print(f"\nRemaining blank property types: {blank_after}")
    if blank_after == 0:
        print("✓ All property types filled!")
    else:
        print(f"⚠️  {blank_after} rows still have no property type")

TESTING WITH OFFICE, SHOP, AND FLAT CLASSIFICATIONS:

Test 1 ✓:
  Expected: Office
  Got: Office
  Preview: VILL/COL: PUPPALGUDA/PUPPAL GUDA W-B: 0-0 SURVEY: 282/P APARTMENT: EON-HYDERABAD FLAT: OFC I EXTENT:...
  Core text: VILL/COL: PUPPALGUDA/PUPPAL GUDA W-B: 0-0 SURVEY: 282/P APARTMENT: EON-HYDERABAD FLAT: OFC I EXTENT:...

Test 2 ✓:
  Expected: Office
  Got: Office
  Preview: VILL/COL: NARSINGI/COMMERCIAL-3 W-B: 0-0 SURVEY: 158/P APARTMENT: JYOTHI OPTIMA FLAT: OFF 4-C EXTENT...
  Core text: VILL/COL: NARSINGI/COMMERCIAL-3 W-B: 0-0 SURVEY: 158/P APARTMENT: JYOTHI OPTIMA FLAT: OFF 4-C EXTENT...

Test 3 ✓:
  Expected: Office
  Got: Office
  Preview: VILL/COL: KOKAPET/COMMERCIAL-3 W-B: 0-0 SURVEY: 107/P 108/P HOUSE: . APARTMENT: LAXMI INFOBAHN, TOWE...
  Core text: VILL/COL: KOKAPET/COMMERCIAL-3 W-B: 0-0 SURVEY: 107/P 108/P HOUSE: . APARTMENT: LAXMI INFOBAHN, TOWE...

Test 4 ✓:
  Expected: Office
  Got: Office
  Preview: VILL/COL: PUPPALGUDA/RESIDENTIAL-2 W-B: 0-0 SURVEY: 285/P APARTM

Classifying property types: 100%|██████████| 45928/45928 [00:54<00:00, 848.94it/s] 



PROPERTY TYPE DISTRIBUTION:
  Plot: 31425 (68.4%)
  House: 14048 (30.6%)
  Flat: 379 (0.8%)
  Office: 60 (0.1%)
  Shop: 11 (0.0%)
  Others: 5 (0.0%)

SAMPLE CLASSIFICATIONS (First 10):

House: VILL/COL: MARYALA/MARYALA VILLAGE W-B: 0-0 HOUSE: 2/87 EXTENT: 311SQ.Yds BUILT: 240SQ. FT HOUSE NO.2...

House: VILL/COL: BHUVANAGIRI/JALEELPURA W-B: 3-5 HOUSE: 3-5-34 EXTENT: 42SQ.Yds BUILT: 200SQ. FT Boundires:...

House: VILL/COL: BOLLE PALLE/BOLLE PALLE VILLAGE W-B: 0-0 HOUSE: 1/70 EXTENT: 87SQ.Yds BUILT: 143SQ. FT HOU...

House: VILL/COL: BHUVANAGIRI/THATA NAGAR W-B: 1-4 HOUSE: 1-4-223 EXTENT: 152SQ.Yds BUILT: 740SQ. FT Boundir...

House: VILL/COL: NANDANAM/NANDANAM VILLAGE W-B: 0-0 HOUSE: 2/54/1 2/54/2 EXTENT: 280SQ.Yds BUILT: 1458SQ. F...

House: VILL/COL: MARYALA/MARYALA VILLAGE W-B: 0-0 HOUSE: 6/65/8 EXTENT: 484SQ.Yds BUILT: 540SQ. FT HOUSE NO...

House: VILL/COL: CHEEKATI MAMIDI/CHEEKATI MAMIDI VILLAGE W-B: 0-0 HOUSE: 1/72 EXTENT: 170SQ.Yds BUILT: 200S...

House: VILL/COL: BHUVANAGIRI/

In [ ]:
df=pd.read_excel(r"D:\Nilesh\Telangana Processing\2024\Yadadri Bhuvanagiri\Yadadri_Bhuvangiri_With_Property_Type.xlsx")

In [7]:
import pandas as pd

# Clean column names (important to avoid KeyError)
df.columns = df.columns.str.strip()

# Exact mapping
mapping = {
    "flat": "Flat",
    "office": "Office",
    "shop": "Shop"
}

# Direct transformation (no extra column)
df["Property_Class"] = (
    df["Property Type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(mapping)
    .fillna("Others")
)

In [8]:
df.columns

Index(['S.No.', 'Description of property', 'Reg.Date Exe.Date Pres.Date',
       'Registration Date', 'Execution Date', 'Presentation Date',
       'Nature & Mkt.Value Con. Value', 'Document type code', 'Document Type',
       'Transaction Type', 'Market Value', 'Consideration Value',
       'Name of Parties Executant(EX) & Claimants(CL)', 'Seller', 'Buyer',
       'Vol/Pg No CD No Doct No/Year', 'Document No', 'District',
       'Sub-Registrar Office', 'VILL/COL', 'W-B', 'SURVEY', 'PLOT', 'HOUSE',
       'APARTMENT', 'BLOCK', 'FLAT', 'EXTENT', 'BUILT', 'Boundires',
       'EXTENT in SqFt', 'BUILT in SqFt', 'Property_type', 'Property Type',
       'Property_Class'],
      dtype='object')

## Rename Column Names 

In [ ]:
# Define mapping from current columns to desired column names
column_mapping = {
    'S.No.': 's.no',
    'Description of property': 'property_description',
    'Registration Date': 'registration_date',
    #'Registration Date': 'transaction_date',
    'Execution Date': 'date_of_execution',
    'Presentation Date': 'presentation_date',
    'Nature & Mkt.Value Con. Value': 'transaction_category',
    'Document type code': 'document_type_code',
    'Document Type': 'document_type',
    'Transaction Type': 'transaction_type',
    'Market Value': 'guideline_value',
    'Consideration Value': 'agreement_price',
    'Name of Parties Executant(EX) & Claimants(CL)': 'party_info',
    'Seller': 'seller_name',
    'Buyer': 'buyer_name',
    'Vol/Pg No CD No Doct No/Year': 'internal_document_number',
    'Document No': 'document_number',
    'District': 'district',
    'Sub-Registrar Office': 'sub_registrar_office_name',
    'VILL/COL': 'village_name',
    'W-B': 'w-b',
    'SURVEY': 'survey_number',
    'PLOT': 'plot_number',
    'HOUSE': 'house_number',
    'APARTMENT': 'project_name',
    'BLOCK': 'block',
    'FLAT': 'unit_number',
    'EXTENT': 'extent_sq_ft',
    'BUILT': 'built_area_sq_ft',
    'Boundires': 'boundaries',
    'EXTENT in SqFt': 'gross_carpet_area_sq_ft',
    'BUILT in SqFt': 'built_area_sq_ft_alternate',
    'Property Type': 'property_type_raw',
    'Property_Class': 'property_type',
    'Final_Area':'final_area',
    'Rate':'rate'
}

# Create mapping for only the columns that exist in your DataFrame
existing_mapping = {col: column_mapping[col] for col in df.columns if col in column_mapping}

# Rename the columns
df = df.rename(columns=existing_mapping)

# Display the renamed columns
print("Renamed columns:")
print(df.columns.tolist())
print(f"\nTotal columns after rename: {len(df.columns)}")

Renamed columns:
['s.no', 'property_description', 'Reg.Date Exe.Date Pres.Date', 'registration_date', 'date_of_execution', 'presentation_date', 'transaction_category', 'document_type_code', 'document_type', 'transaction_type', 'market_value', 'agreement_price', 'party_info', 'seller_name', 'buyer_name', 'internal_document_number', 'document_number', 'district', 'sub_registrar_office_name', 'village_name', 'w-b', 'survey_number', 'plot_number', 'house_number', 'project_name', 'block', 'unit_number', 'extent_sq_ft', 'built_area_sq_ft', 'boundaries', 'gross_carpet_area_sq_ft', 'built_area_sq_ft_alternate', 'Property_type', 'property_type_raw', 'property_type']

Total columns after rename: 35


In [16]:
df=pd.read_excel(r"D:\Nilesh\Telangana Processing\2024 Telangana\Rate_Finalize\Telangana_Final_With_Rate.xlsx")

In [17]:
df['village_name_Cleaned'] = df['village_name'].copy()

In [21]:
col_to_move = 'village_name_Cleaned'
after_col = 'village_name'

# remove column
temp = df.pop(col_to_move)

# get position of reference column
position = df.columns.get_loc(after_col)

# insert after it
df.insert(position + 1, col_to_move, temp)

In [23]:
df.to_excel("Telangana_Final_With_Rate_Updated.xlsx",index=False)

In [10]:
# ====== COLUMN NAME ======
col = 'project_name'   # <-- change if your column name is different
new_col = f'{col}_cleaned'  # Name of the new column

# Abbreviations to keep capital
abbreviations = ['SLG', 'SV', 'SR', 'SS', 'GK']

def clean_name(text):
    if pd.isna(text):
        return text
    
    text = str(text)
    
    # Remove quotes
    text = re.sub(r"[\"']", "", text)
    
    # Remove extra spaces
    text = text.strip()
    
    # Convert to proper case
    text = text.lower().title()
    
    # Fix 's
    text = re.sub(r"'S\b", "'s", text)
    
    # Fix abbreviations
    words = text.split()
    words = [w.upper() if w.upper() in abbreviations else w for w in words]
    
    return " ".join(words)

# Create new column with cleaned names
df[new_col] = df[col].apply(clean_name)

In [11]:
print(df['project_name'].head(10))

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
Name: project_name, dtype: object


In [2]:
df=pd.read_excel(r"D:\Nilesh\Telangana Processing\2024 Telangana\Rate_Finalize\RangaReddy_Final(Manual Done by Nilesh).xlsx")
df.head(2)

,s.no,property_description,registration_date,transaction_date,date_of_execution,presentation_date,transaction_category,document_type_code,document_type,transaction_type,...,block,unit_number,extent_sq_ft,built_area_sq_ft,boundaries,gross_carpet_area_sq_ft,built_area_sq_ft_alternate,Property_type_Filled,property_type_raw,property_type
0,1.0,VILL/COL: ABDULLAPUR/ABDULLAPUR W-B: 0-1 SURVE...,(R) 31-12-2024 (E) 31-12-2024 (P) 31-12-2024,31-12-2024,31-12-2024,31-12-2024,0111 AGREEMENT OF SALE CUM GPA Mkt.Value:Rs. 4...,111.0,AGREEMENT OF SALE CUM GPA,Sales,...,NaN,NaN,208SQ.Yds,NaN,[N]: 30' WIDE ROAD [S] PLOT NO 251 [E]: PLOT N...,1872.0,NaN,NaN,Plot,Others
1,1.0,VILL/COL: ABDULLAPUR/ABDULLAPUR W-B: 0-1 SURVE...,(R) 31-12-2024 (E) 31-12-2024 (P) 31-12-2024,31-12-2024,31-12-2024,31-12-2024,0111 AGREEMENT OF SALE CUM GPA Mkt.Value:Rs. 4...,111.0,AGREEMENT OF SALE CUM GPA,Sales,...,NaN,NaN,200SQ.Yds,NaN,[N]: 30' WIDE ROAD [S] PLOT NO 332 [E]: PLOT N...,1800.0,NaN,NaN,Plot,Others


In [10]:
df = merged_df

## Rate

In [13]:
df.columns

Index(['s.no', 'property_description', 'Reg.Date Exe.Date Pres.Date',
       'registration_date', 'date_of_execution', 'presentation_date',
       'transaction_category', 'document_type_code', 'document_type',
       'transaction_type', 'market_value', 'agreement_price', 'party_info',
       'seller_name', 'buyer_name', 'internal_document_number',
       'document_number', 'district', 'sub_registrar_office_name',
       'village_name', 'w-b', 'survey_number', 'plot_number', 'house_number',
       'project_name', 'project_name_cleaned', 'block', 'unit_number',
       'extent_sq_ft', 'built_area_sq_ft', 'boundaries',
       'gross_carpet_area_sq_ft', 'built_area_sq_ft_alternate',
       'property_type_raw', 'property_type', 'Final_Area', 'Rate'],
      dtype='object')

In [12]:
import pandas as pd
import numpy as np

# Clean column names (important)
df.columns = df.columns.str.strip()

def calculate_area(row):
    prop = str(row["property_type"]).strip().lower()
    built = row.get("built_area_sq_ft_alternate", np.nan)
    extent = row.get("gross_carpet_area_sq_ft", np.nan)

    # Flat / Office / Shop
    if prop in ["flat", "office", "shop"]:
        return built

    # Others
    else:
        return extent


# Apply area selection
df["Final_Area"] = df.apply(calculate_area, axis=1)

# Rate calculation
df["Rate"] = df["agreement_price"] / df["Final_Area"]

In [14]:
df.to_excel("Telangana_Final_With_Rate.xlsx",index=False)

### Columns Uniformnity Checked

In [5]:
import os
import pandas as pd

# =========================
# SETTINGS
# =========================
folder_path = r"D:\Nilesh\Telangana Processing\2024 Telangana\Rate_Finalize"   # change this
sheet_name = 0   # first sheet; change if needed, e.g. "Sheet1"

# Optional cleaning of column names
def clean_columns(cols):
    return [str(col).strip() for col in cols]

# =========================
# GET EXCEL FILES
# =========================
excel_files = [
    f for f in os.listdir(folder_path)
    if f.lower().endswith((".xlsx", ".xls", ".xlsm"))
]

if not excel_files:
    print("No Excel files found in the folder.")
    exit()

# =========================
# READ COLUMNS FROM FILES
# =========================
file_columns = {}

for file in excel_files:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, nrows=0)
        cols = clean_columns(df.columns.tolist())
        file_columns[file] = cols
    except Exception as e:
        print(f"Error reading {file}: {e}")

if not file_columns:
    print("No readable Excel files found.")
    exit()

# =========================
# COMPARE COLUMNS
# =========================
files = list(file_columns.keys())
base_file = files[0]
base_cols = file_columns[base_file]
base_set = set(base_cols)

all_same = True

print(f"\nReference file: {base_file}")
print(f"Reference columns ({len(base_cols)}): {base_cols}\n")

for file, cols in file_columns.items():
    current_set = set(cols)

    missing_cols = list(base_set - current_set)
    extra_cols = list(current_set - base_set)

    if current_set != base_set:
        all_same = False
        print(f"File: {file}")
        print("Status: DIFFERENT")

        if missing_cols:
            print("Missing columns compared to reference:")
            print(missing_cols)

        if extra_cols:
            print("Extra columns compared to reference:")
            print(extra_cols)

        print("-" * 60)

if all_same:
    print("All Excel files have the same columns.")
else:
    print("Some files have different columns.")

# =========================
# OPTIONAL: SAVE SUMMARY TO EXCEL
# =========================
summary_rows = []

for file, cols in file_columns.items():
    current_set = set(cols)
    missing_cols = list(base_set - current_set)
    extra_cols = list(current_set - base_set)

    summary_rows.append({
        "file_name": file,
        "same_as_reference": current_set == base_set,
        "missing_columns": ", ".join(missing_cols),
        "extra_columns": ", ".join(extra_cols),
        "total_columns": len(cols)
    })

summary_df = pd.DataFrame(summary_rows)
output_path = os.path.join(folder_path, "column_comparison_summary.xlsx")
summary_df.to_excel(output_path, index=False)

print(f"\nSummary file saved at:\n{output_path}")


Reference file: Hydrabad_Final(Manual_Done_By_Nilesh).xlsx
Reference columns (35): ['s.no', 'property_description', 'Reg.Date Exe.Date Pres.Date', 'registration_date', 'date_of_execution', 'presentation_date', 'transaction_category', 'document_type_code', 'document_type', 'transaction_type', 'market_value', 'agreement_price', 'party_info', 'seller_name', 'buyer_name', 'internal_document_number', 'document_number', 'district', 'sub_registrar_office_name', 'village_name', 'w-b', 'survey_number', 'plot_number', 'house_number', 'project_name', 'project_name_cleaned', 'block', 'unit_number', 'extent_sq_ft', 'built_area_sq_ft', 'boundaries', 'gross_carpet_area_sq_ft', 'built_area_sq_ft_alternate', 'property_type_raw', 'property_type']

All Excel files have the same columns.

Summary file saved at:
D:\Nilesh\Telangana Processing\2024 Telangana\Rate_Finalize\column_comparison_summary.xlsx
